# CS3807 Deep Learning Laboratory - Part 6: Grad-CAM, Internal Maps, Qualitative Analysis
Grad-CAM heatmaps, internal channel/spatial attention maps, and qualitative comparisons.

In [ ]:
# Cell 1: Imports, models, sample images
import os, pickle
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
os.makedirs('results/figures', exist_ok=True)
with open('results/all_metrics.pkl', 'rb') as f:
    saved = pickle.load(f)
best = saved['best_model']
with open('results/data_splits.pkl', 'rb') as f:
    splits = pickle.load(f)
X_test, y_test, class_names = splits['X_test'], splits['y_test'], splits['class_names']
y_true = np.argmax(y_test, axis=1)
baseline = keras.models.load_model('results/models/A0_Baseline_model.h5', compile=False)
best_model = keras.models.load_model(f"results/models/{best}_model.h5", compile=False)
rng = np.random.default_rng(0)
sample_idx = rng.choice(len(X_test), size=6, replace=False)
print(f"✓ Baseline + best ({best}) loaded; {len(sample_idx)} sample images")

In [ ]:
# Cell 2: Grad-CAM utility + heatmaps (baseline vs best)
def grad_cam(model, img, layer_name='conv5_block3_out'):
    gm = keras.Model(model.input, [model.get_layer(layer_name).output, model.output])
    with tf.GradientTape() as tape:
        conv, pred = gm(tf.expand_dims(img, 0))
        loss = pred[:, tf.argmax(pred[0])]
    g = tape.gradient(loss, conv)[0]
    w = tf.reduce_mean(g, axis=(0, 1))
    cam = tf.reduce_sum(conv[0] * w, axis=-1).numpy()
    cam = np.maximum(cam, 0); cam /= (cam.max() + 1e-8)
    return cam
fig, axes = plt.subplots(6, 4, figsize=(10, 14))
for r, i in enumerate(sample_idx):
    img = X_test[i]
    cb, ca = grad_cam(baseline, img), grad_cam(best_model, img)
    axes[r, 0].imshow(img.astype(np.uint8)); axes[r, 0].set_title(f"True: {class_names[y_true[i]]}")
    axes[r, 1].imshow(cb, cmap='jet'); axes[r, 1].set_title('Baseline Grad-CAM')
    axes[r, 2].imshow(img.astype(np.uint8))
    axes[r, 2].imshow(ca, cmap='jet', alpha=0.45); axes[r, 2].set_title(f'{best} Grad-CAM')
    axes[r, 3].imshow(ca, cmap='jet'); axes[r, 3].set_title('Best heatmap')
    for c in range(4): axes[r, c].axis('off')
fig.tight_layout(); fig.savefig('results/figures/gradcam_heatmaps.png', dpi=150); plt.show()

In [ ]:
# Cell 3: Internal channel attention vectors + spatial attention maps
att_layers = [l for l in best_model.layers if 'attention' in l.name.lower() or
              any(k in l.name.lower() for k in ['channel', 'spatial', 'eca', 'cbam', 'se'])]
print('Attention layers found:', [l.name for l in att_layers] or 'none by name - using Grad-CAM maps')
extractor = keras.Model(best_model.input, [l.output for l in att_layers]) if att_layers else None
img = X_test[sample_idx[0]]
n = max(len(att_layers), 1)
fig, axes = plt.subplots(1, n + 1, figsize=(4 * (n + 1), 4))
axes = np.atleast_1d(axes)
axes[0].imshow(img.astype(np.uint8)); axes[0].set_title('Input'); axes[0].axis('off')
if extractor is not None:
    for ax, out, l in zip(axes[1:], extractor(tf.expand_dims(img, 0)), att_layers):
        m = tf.reduce_mean(out[0], axis=-1).numpy()
        ax.imshow(m, cmap='viridis'); ax.set_title(l.name); ax.axis('off')
    fig.suptitle('Internal Spatial Maps')
    fig.tight_layout(); fig.savefig('results/figures/internal_spatial_maps.png', dpi=150); plt.show()
    vec = None
    for out in extractor(tf.expand_dims(img, 0)):
        v = tf.reduce_mean(out[0], axis=(0, 1)).numpy()
        if vec is None or v.size < vec.size: vec = v
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.bar(range(vec.size), vec, color='darkorange')
    ax.set_title('Internal Channel Attention Weights'); ax.set_xlabel('channel')
    fig.tight_layout(); fig.savefig('results/figures/internal_channel_maps.png', dpi=150); plt.show()
else:
    fig.savefig('results/figures/internal_spatial_maps.png', dpi=150)

In [ ]:
# Cell 4: Qualitative comparison grid + attention-weight distributions
names = ['A0_Baseline', best]
models = [baseline, best_model]
preds = [np.argmax(m.predict(X_test[sample_idx], verbose=0), axis=1) for m in models]
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for c, i in enumerate(sample_idx):
    for r in range(2):
        ok = preds[r][c] == y_true[i]
        axes[r, c].imshow(X_test[i].astype(np.uint8))
        axes[r, c].set_title(f"{'OK' if ok else 'WRONG'}: {class_names[preds[r][c]]}",
                             color='green' if ok else 'red')
        axes[r, c].axis('off')
    axes[0, c].set_ylabel('Baseline', rotation=0, labelpad=25)
    axes[1, c].set_ylabel('Best', rotation=0, labelpad=25)
fig.tight_layout(); fig.savefig('results/figures/qualitative_comparison.png', dpi=150); plt.show()
fig, ax = plt.subplots(figsize=(8, 4))
for name, m in zip(names, models):
    p = m.predict(X_test, batch_size=32, verbose=0)
    ax.hist(np.max(p, axis=1), bins=30, alpha=0.5, label=name)
ax.set_xlabel('Top-1 confidence'); ax.set_ylabel('count'); ax.legend()
ax.set_title('Prediction Confidence Distributions')
fig.tight_layout(); fig.savefig('results/figures/attention_weight_distributions.png', dpi=150)
plt.show()

In [ ]:
# Cell 5: Misclassified-case analysis (baseline wrong, best right)
pb = np.argmax(baseline.predict(X_test, batch_size=32, verbose=0), axis=1)
pa = np.argmax(best_model.predict(X_test, batch_size=32, verbose=0), axis=1)
fixed = np.where((pb != y_true) & (pa == y_true))[0][:6]
print(f"Cases fixed by attention: {(pb != y_true).sum()} baseline errors -> "
      f"{(pa != y_true).sum()} best errors; showing {len(fixed)}")
fig, axes = plt.subplots(1, max(len(fixed), 1), figsize=(3 * max(len(fixed), 1), 3.5))
axes = np.atleast_1d(axes)
for ax, i in zip(axes, fixed):
    ax.imshow(X_test[i].astype(np.uint8))
    ax.set_title(f"True:{class_names[y_true[i]]}\nBase:{class_names[pb[i]]}->Best:{class_names[pa[i]]}")
    ax.axis('off')
fig.tight_layout(); fig.savefig('results/figures/misclassified_analysis.png', dpi=150); plt.show()
print("✓ All Part-6 figures saved")

In [ ]:
# Cell 6: Spatial attention maps Ms^P2, Ms^P3, Ms^P4 (heatmap + overlay on input)
pos_models = [('P2', 'results/models/B2_P2_model.h5'), ('P3', 'results/models/B3_P3_model.h5'),
              ('P4', 'results/models/B4_P4_model.h5')]
img = X_test[sample_idx[0]]
avail = [(p, mp) for p, mp in pos_models if os.path.exists(mp)]
if not avail:
    print("  ! B2/B3/B4 weights missing - run Part 3 Experiment 2 first")
else:
    fig, axes = plt.subplots(len(avail), 3, figsize=(9, 3.5 * len(avail)))
    axes = np.atleast_2d(axes)
    for r, (p, mp) in enumerate(avail):
        m = keras.models.load_model(mp, compile=False)
        sp = next(l for l in m.layers if 'spatial' in l.name.lower())
        fmap = keras.Model(m.input, sp.output)(tf.expand_dims(img, 0))[0].numpy()
        heat = fmap.mean(axis=-1)
        heat = (heat - heat.min()) / (heat.max() - heat.min() + 1e-8)
        heat = tf.image.resize(heat[..., None], (224, 224)).numpy()[..., 0]
        axes[r, 0].imshow(img.astype(np.uint8)); axes[r, 0].set_title(f"Input ({p})")
        axes[r, 1].imshow(heat, cmap='jet'); axes[r, 1].set_title(f"Ms^{p} heatmap")
        axes[r, 2].imshow(img.astype(np.uint8))
        axes[r, 2].imshow(heat, cmap='jet', alpha=0.45); axes[r, 2].set_title(f"Ms^{p} overlay")
        for cc in range(3): axes[r, cc].axis('off')
    fig.suptitle('Spatial Attention at P2 / P3 / P4')
    fig.tight_layout(); fig.savefig('results/figures/spatial_maps_P2_P3_P4.png', dpi=150); plt.show()

In [ ]:
# Cell 7: Feature map BEFORE vs AFTER attention (best model, first attention block)
idx = next(i for i, l in enumerate(best_model.layers)
             if 'attention' in type(l).__name__.lower() or 'attention' in l.name.lower())
prev, attn = best_model.layers[idx - 1], best_model.layers[idx]
ext = keras.Model(best_model.input, [prev.output, attn.output])
fb, fa = ext(tf.expand_dims(img, 0))
before = fb[0].numpy().mean(axis=-1); after = fa[0].numpy().mean(axis=-1)
before = (before - before.min()) / (before.max() - before.min() + 1e-8)
after = (after - after.min()) / (after.max() - after.min() + 1e-8)
fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
axes[0].imshow(X_test[sample_idx[0]].astype(np.uint8)); axes[0].set_title('Input')
axes[1].imshow(before, cmap='viridis'); axes[1].set_title(f'Before attention ({prev.name})')
axes[2].imshow(after, cmap='viridis'); axes[2].set_title(f'After attention ({attn.name})')
for ax in axes: ax.axis('off')
fig.tight_layout(); fig.savefig('results/figures/feature_before_after_attention.png', dpi=150)
plt.show()

In [ ]:
# Cell 8: Channel-attention weights, top-K bars (K=20)
K = 20
ch = next((l for l in best_model.layers if any(k in l.name.lower()
           for k in ['channel_attention', 'eca', 'se_attention', 'cbam'])), None)
vec = keras.Model(best_model.input, ch.output)(tf.expand_dims(img, 0))[0].numpy().mean(axis=(0, 1)) \
    if ch is not None else \
    keras.Model(best_model.input, best_model.get_layer('conv4_block6_out').output)(tf.expand_dims(img, 0))[0].numpy().mean(axis=(0, 1))
topk = np.argsort(vec)[::-1][:K]
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(range(K), vec[topk], color='darkorange')
ax.set_xticks(range(K)); ax.set_xticklabels([str(int(t)) for t in topk], rotation=45, fontsize=8)
ax.set_xlabel(f'top-{K} channel index'); ax.set_ylabel('attention weight')
ax.set_title(f'Top-{K} Channel-Attention Weights ({ch.name if ch is not None else "fallback"})')
fig.tight_layout(); fig.savefig('results/figures/channel_topk_weights.png', dpi=150); plt.show()

In [ ]:
# Cell 9: Self-attention response (A4_Self model)
sp = 'results/models/A4_Self_model.h5'
if not os.path.exists(sp):
    print("  ! A4_Self weights missing - run Part 3 Experiment 1 first")
else:
    sm = keras.models.load_model(sp, compile=False)
    sa = next(l for l in sm.layers if 'self_attention' in l.name.lower())
    resp = keras.Model(sm.input, sa.output)(tf.expand_dims(img, 0))[0].numpy()
    rmap = resp.mean(axis=-1)
    rmap = (rmap - rmap.min()) / (rmap.max() - rmap.min() + 1e-8)
    rmap = tf.image.resize(rmap[..., None], (224, 224)).numpy()[..., 0]
    fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
    axes[0].imshow(img.astype(np.uint8)); axes[0].set_title('Input')
    axes[1].imshow(rmap, cmap='plasma'); axes[1].set_title('Self-attention response')
    axes[2].imshow(img.astype(np.uint8))
    axes[2].imshow(rmap, cmap='plasma', alpha=0.45); axes[2].set_title('Response overlay')
    for ax in axes: ax.axis('off')
    fig.tight_layout(); fig.savefig('results/figures/self_attention_response.png', dpi=150)
    plt.show()

---

**Part 6 outputs (10 figures) - full-run totals: Part 4 (2) + Part 5 (9) + Part 6 (10) + Part 1 (1) = 22 visualizations:**
- `confusion_matrix_baseline.png`, `confusion_matrix_best.png`, `exp1_type_accuracy.png`,
  `exp2_position_accuracy.png`, `training_history_curves.png`, `training_loss_curves.png`,
  `f1_comparison.png`, `order_ablation_exp4.png`, `kernel_ablation_exp5.png`,
  `accuracy_vs_params.png`, `f1_vs_infertime.png`, `gradcam_heatmaps.png`,
  `internal_channel_maps.png`, `internal_spatial_maps.png`, `qualitative_comparison.png`,
  `attention_weight_distributions.png`, `misclassified_analysis.png`,
  `spatial_maps_P2_P3_P4.png`, `feature_before_after_attention.png`,
  `channel_topk_weights.png`, `self_attention_response.png`, `01_sample_images.png`

**Lab complete.**